In [1]:
import pandas as pd

In [2]:
import torch

In [3]:
from torch import nn, optim

In [4]:
import torch.nn.functional as F

In [5]:
from string import punctuation

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
from nltk.tokenize import word_tokenize

In [9]:
df = pd.read_csv("C:\\git\\dados\\nlp\\news_sentiment_analysis.csv", encoding="utf-8")

In [10]:
# dicionario_sentimentos = {}
# indice_sentimentos = 0
# for sentimento in df["Sentiment"]:
#     if sentimento not in dicionario_sentimentos:
#         dicionario_sentimentos[sentimento] = indice_sentimentos
#         indice_sentimentos += 1
# print(dicionario_sentimentos)

In [11]:
# df["sentimento_number"] = [dicionario_sentimentos[s] for s in df["Sentiment"]]
# df

In [12]:
# sentimento_numero = torch.tensor(df["sentimento_number"])

# # Aplicando One-Hot Encoding
# df["sentimento_one_hot"] = F.one_hot(sentimento_numero, num_classes=3).tolist()

In [13]:
# df["sentimento_numerico"] = df["Sentiment"].map( {"positive": [1, 0, 0], "negative": [0, 0, 1], "neutral": [0, 1, 0]} )

In [14]:
df = df[["Description"]]

In [17]:
def tokenizar_limpar_texto( texto : str ) -> list:
    trans_table = str.maketrans("—–", "  ", punctuation)
    texto_limpo = texto.lower().translate( trans_table ).replace("  ", " ")
    tokens = word_tokenize(texto_limpo, language='english')
    tokens_limpos = []
    for token in tokens:
        if len(token) <= 15:
            tokens_limpos.append( token )
    return tokens_limpos

In [18]:
df["TokensLimpos"] = df["Description"].apply( tokenizar_limpar_texto )
df

,Description,TokensLimpos
0,"ST. GEORGE — Kaitlyn Larson, a first-year teac...","[st, george, kaitlyn, larson, a, firstyear, te..."
1,"Harare, Zimbabwe – Local businesses are grappl...","[harare, zimbabwe, local, businesses, are, gra..."
2,(marketscreener.com) Billionaire Elon Musk has...,"[billionaire, elon, musk, has, donated, to, a,..."
3,(marketscreener.com) A U.S. trade regulator on...,"[a, us, trade, regulator, on, fridayannounced,..."
4,4.5 million households in the U.S. have solar ...,"[45, million, households, in, the, us, have, s..."
...,...,...
3495,QRG Capital Management Inc. increased its stak...,"[qrg, capital, management, inc, increased, its..."
3496,QRG Capital Management Inc. bought a new posit...,"[qrg, capital, management, inc, bought, a, new..."
3497,QRG Capital Management Inc. boosted its stake ...,"[qrg, capital, management, inc, boosted, its, ..."
3498,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...","[westford, mass, july, 18, 2024, prnewswire, a..."


In [20]:
dicionario = {}
indice_dicionario = 0
for tokens in df["TokensLimpos"]:
    for token in tokens:
        if token not in dicionario:
            dicionario[token] = indice_dicionario
            indice_dicionario += 1
print(dicionario)

{'st': 0, 'george': 1, 'kaitlyn': 2, 'larson': 3, 'a': 4, 'firstyear': 5, 'teacher': 6, 'at': 7, 'pine': 8, 'view': 9, 'high': 10, 'school': 11, 'recently': 12, 'received': 13, 'the': 14, 'best': 15, 'in': 16, 'state': 17, 'award': 18, 'for': 19, 'business': 20, 'leadership': 21, 'course': 22, '’': 23, 's': 24, 'students': 25, 'grades': 26, '1012': 27, 'had': 28, 'highest': 29, 'passing': 30, 'rate': 31, 'on': 32, 'youscience': 33, 'exam': 34, 'during': 35, 'spring': 36, 'and': 37, 'fall': 38, 'semesters': 39, 'of': 40, '2023': 41, 'is': 42, 'similar': 43, 'to': 44, 'final': 45, 'exams': 46, '8230': 47, 'harare': 48, 'zimbabwe': 49, 'local': 50, 'businesses': 51, 'are': 52, 'grappling': 53, 'with': 54, 'severe': 55, 'liquidity': 56, 'crunch': 57, 'which': 58, 'limiting': 59, 'billionaire': 60, 'elon': 61, 'musk': 62, 'has': 63, 'donated': 64, 'super': 65, 'pac': 66, 'working': 67, 'elect': 68, 'former': 69, 'us': 70, 'president': 71, 'donald': 72, 'trump': 73, 'bloomberg': 74, 'reporte

In [25]:
# converter os tokens em numeros
tokens_numeros = []
MAIOR_FRASE = 0
for tokens in df["TokensLimpos"]:
    numeros = []
    if len(tokens) > MAIOR_FRASE:
        MAIOR_FRASE = len(tokens)
    for token in tokens:
        numero = dicionario.get(token, -1)  # -1 ==> <NAO ENCONTRADO>
        numeros.append( numero )
    tokens_numeros.append(numeros)
df["TokenNumeros"] = tokens_numeros
print("Mario Frase==>", MAIOR_FRASE)

Mario Frase==> 113


In [32]:
# Fazer o Padding
tokens_padded = []
saida = []
for tokens in df["TokenNumeros"]:
    number_paddings = MAIOR_FRASE - len(tokens) - 1 
    paddings = [0 for _ in range(number_paddings)]
    tokens_padded.append( paddings + tokens[:-1] )
    saida.append(tokens[-1])
    print(paddings + tokens)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 4, 20, 21, 22, 3, 23, 24, 25, 16, 26, 27, 28, 14, 29, 30, 31, 32, 14, 33, 34, 35, 14, 36, 37, 38, 39, 40, 41, 14, 33, 34, 42, 43, 44, 45, 46, 47]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 48, 49, 50, 51, 52, 53, 54, 4, 55, 56, 57, 58, 42, 59]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 60, 61, 62, 63, 64, 44, 4, 65, 66, 67, 44, 68, 69, 70, 71, 72, 73, 74

In [ ]:
tokens_padded

"a car wash was closed" ==> [10, 20, 50, 38, 47]
1 instancia ==> [ [0], [0], [0], [10], [20], [50], [38], [47] ] 

"a great car sale is occuring in inner country" ==> [10, 30, 20, 79, 104, 215, 327, 68]
1 instancia ==> [ [10], [30], [20], [79], [104], [215], [327], [68] ] 

In [29]:
X = torch.tensor( tokens_padded, dtype=torch.float32)
X

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 4.5000e+01, 4.6000e+01,
         4.7000e+01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 5.8000e+01, 4.2000e+01,
         5.9000e+01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 9.3000e+01, 9.4000e+01,
         6.6000e+01],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 1.9683e+04, 9.3900e+02,
         4.7000e+01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 7.7860e+03, 1.6000e+01,
         5.6300e+02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 5.3000e+02, 9.6570e+03,
         4.7000e+01]])

In [37]:
Y = torch.tensor(saida, dtype=torch.float32)
Y = Y.reshape(-1, 1)
Y

tensor([[ 47.],
        [ 59.],
        [ 66.],
        ...,
        [ 47.],
        [563.],
        [ 47.]])

In [36]:
vocabulario = vetorizador.get_feature_names_out()

In [37]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.20, random_state = 100)

In [24]:
modelo = nn.Sequential( 
    nn.Linear(in_features=MAIOR_FRASE, out_features=3),
    # nn.Linear(in_features=MAX_PALAVRAS, out_features=512),
    # nn.Linear(in_features=512, out_features=1),
    # nn.Sigmoid()
)
modelo

NameError: name 'MAIOR_FRASE' is not defined

In [39]:
for param in modelo.parameters():
    print(param)

Parameter containing:
tensor([[-0.0017, -0.0127,  0.0133,  ..., -0.0180,  0.0085, -0.0180],
        [ 0.0164, -0.0174, -0.0132,  ..., -0.0041, -0.0122,  0.0047],
        [ 0.0060,  0.0074, -0.0087,  ..., -0.0013,  0.0121, -0.0080]],
       requires_grad=True)
Parameter containing:
tensor([-0.0053, -0.0078,  0.0098], requires_grad=True)


In [40]:
funcao_perda = nn.CrossEntropyLoss()

In [41]:
otimizador = optim.SGD( modelo.parameters(), lr = 0.1)

In [43]:
epocas = 500
for epoca in range(epocas):
    y_hat = modelo( X_train )
    loss = funcao_perda( y_hat, Y_train )
    if epoca % 100 == 0: 
        print("Loss: ", loss)
    loss.backward()
    otimizador.step()

Loss:  tensor(6.9939e-07, grad_fn=<DivBackward1>)
Loss:  tensor(1.0892e-06, grad_fn=<DivBackward1>)
Loss:  tensor(1.7758e-06, grad_fn=<DivBackward1>)
Loss:  tensor(2.6098e-06, grad_fn=<DivBackward1>)
Loss:  tensor(3.1412e-06, grad_fn=<DivBackward1>)


In [24]:
# acertos = 0 

In [25]:
# texto = df["TextoLimpo"].loc[3496]


In [26]:
# teste_vetorizador = CountVectorizer(vocabulary = vocabulario)

In [27]:
# X_test = teste_vetorizador.fit_transform( df["TextoLimpo"] )
# X_test.toarray()

In [28]:
# teste_vetorizador.get_feature_names_out()

In [29]:
# X_test = torch.tensor( X_test.toarray(), dtype=torch.float32 ) 
# X_test.shape

In [44]:
y_hat_test = modelo( X_test )
y_hat_test

tensor([[-216.9860,  406.9200, -190.0058],
        [ -37.4787,  292.7585, -255.3878],
        [ 966.7238, -312.6717, -654.2052],
        ...,
        [ 151.8871, -184.6499,   32.7825],
        [ 233.4522, -119.4969, -114.0457],
        [ 310.7687, -197.0166, -113.6648]], grad_fn=<AddmmBackward0>)

In [52]:
acertos = 0
for idx, y_hat in enumerate(y_hat_test):
    y = Y_test[idx]
    y_index = torch.argmax( y )
    y_hat_index = torch.argmax( y_hat )
    # resultado_oficial = Y_test[idx].item()
    # resultado_previsto = 0
    # if y_hat.item() >= 0.5:
    #     resultado_previsto = 1
    if y_index == y_hat_index:
        acertos = acertos + 1
    print(f"{y_hat}=>{y_hat_index} | {y} => {y_index}")
print("Porcentagem de acerto: ",  (acertos / len(y_hat_test)) * 100 )    

tensor([-216.9860,  406.9200, -190.0058], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ -37.4787,  292.7585, -255.3878], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ 966.7238, -312.6717, -654.2052], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 519.2092, -702.2241,  183.1143], grad_fn=<UnbindBackward0>)=>0 | tensor([0., 0., 1.]) => 2
tensor([ 636.1210, -227.5637, -408.7502], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 239.0228, -221.9166,  -16.8874], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([-272.2610,  671.8472, -399.9415], grad_fn=<UnbindBackward0>)=>1 | tensor([0., 1., 0.]) => 1
tensor([ 309.6625, -127.5140, -182.2730], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 868.5778, -622.9871, -245.8302], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) => 0
tensor([ 1172.3693, -1056.6765,  -115.8165], grad_fn=<UnbindBackward0>)=>0 | tensor([1., 0., 0.]) =>